
# Commercial Intelligence & Profitability Analytics

## 03 — Análisis del desempeño comercial

**Empresa:** Distribuidora Cordillera (datos ficticios)  
**Tecnologías:** SQLite, SQL avanzado, Jupyter Notebook  
**Fuente:** Modelo dimensional implementado en `cordillera_analytics.sqlite`

### Objetivo general

Evaluar el desempeño comercial de Distribuidora Cordillera mediante el análisis de su evolución temporal, composición de la actividad comercial, comportamiento de clientes y desempeño de productos y categorías.

El propósito es identificar patrones, variaciones y concentraciones relevantes para apoyar la interpretación de los resultados comerciales y fundamentar análisis posteriores de rentabilidad.

### Alcance

El análisis se organizará en tres dimensiones principales:

1. **Evolución comercial:** comportamiento mensual de pedidos, importes netos, costos y márgenes brutos estimados.
2. **Clientes y segmentos:** distribución de la actividad comercial, frecuencia de compra y comportamiento de clientes.
3. **Productos y categorías:** participación, volumen y desempeño comercial del catálogo.

### Metodología

Se utilizarán las vistas validadas durante el modelado dimensional:

- `fact_detalle_pedidos`
- `dim_cliente`
- `dim_producto`
- `dim_fecha`

Las consultas se desarrollarán principalmente mediante agregaciones, CTE y funciones de ventana.

Cada análisis incluirá una definición de indicadores, resultados y una interpretación de los hallazgos relevantes.

### Criterios de interpretación

Los indicadores principales considerarán pedidos con estado `entregado`, salvo que se indique expresamente lo contrario.

El importe neto de pedidos entregados no representa necesariamente facturación emitida ni ingresos efectivamente cobrados.

Los márgenes brutos calculados constituyen estimaciones basadas en los costos registrados en las líneas de pedido.

Los resultados deben interpretarse considerando las limitaciones de los datos y las reglas de negocio establecidas durante la auditoría y el modelado dimensional.


In [117]:
%load_ext sql
%sql sqlite:///../data/cordillera_analytics.sqlite

The sql extension is already loaded. To reload it, use:
  %reload_ext sql



## 3.1 Evolución del desempeño comercial

### 3.1.1 Indicadores comerciales mensuales

#### Objetivo

Analizar la evolución mensual de la actividad comercial mediante indicadores de pedidos entregados, importes netos, costos asociados y márgenes brutos estimados.

#### Metodología

Se relacionarán `fact_detalle_pedidos` y `dim_fecha` mediante la fecha de creación del pedido.

Se considerarán exclusivamente pedidos en estado `entregado`.

Los resultados se agruparán por período `anio_mes`, conservando el orden cronológico.

**Criterio temporal:** los importes corresponden a pedidos actualmente registrados como entregados, agrupados según su fecha de creación (`fecha_pedido`), no según su fecha efectiva de entrega.

Esta distinción deberá considerarse al interpretar la evolución mensual.


In [118]:
%%sql

SELECT
    anio_mes,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.costo_linea_clp) AS costo_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp
FROM fact_detalle_pedidos AS t
LEFT JOIN dim_fecha AS f
    ON t.fecha_pedido = f.fecha
WHERE t.estado = 'entregado'
GROUP BY anio_mes
ORDER BY anio_mes ASC
;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

anio_mes,pedidos_entregados,importe_neto_entregado_clp,costo_entregado_clp,margen_bruto_estimado_clp
2024-01,442,125743928,90448000,35295928
2024-02,307,92063024,66400800,25662224
2024-03,403,117070132,84207100,32863032
2024-04,365,105131436,76156800,28974636
2024-05,376,113535858,81961700,31574158
2024-06,334,96920786,70271700,26649086
2024-07,366,111328472,80432800,30895672
2024-08,387,111880944,80746900,31134044
2024-09,450,128508756,92507800,36000956
2024-10,425,125949504,90548400,35401104


In [119]:
%%sql

WITH resumen_mensual AS (
SELECT
    anio_mes,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.costo_linea_clp) AS costo_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp
FROM fact_detalle_pedidos AS t
LEFT JOIN dim_fecha AS f
    ON t.fecha_pedido = f.fecha
WHERE t.estado = 'entregado'
GROUP BY anio_mes
), 
comparacion_mensual AS (
SELECT
    anio_mes,
    pedidos_entregados,
    importe_neto_entregado_clp,
    ROUND(
        1.0 * importe_neto_entregado_clp / NULLIF(pedidos_entregados, 0)
        ,2) AS ticket_promedio_clp,
    ROUND(
        100.0 * (pedidos_entregados - LAG(pedidos_entregados) OVER (ORDER BY anio_mes) ) /  
        NULLIF(LAG(pedidos_entregados) OVER (ORDER BY anio_mes) , 0)
        , 2) AS variacion_pedidos_mom_pct,
    ROUND(
        100.0 * (importe_neto_entregado_clp - LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes) ) /  
        NULLIF(LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes), 0)
        , 2) AS variacion_importe_mom_pct,
   ROUND(
        AVG(importe_neto_entregado_clp) OVER(
        ORDER BY anio_mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
        , 2) AS promedio_movil_importe_3m
FROM resumen_mensual
)
SELECT
    anio_mes,
    pedidos_entregados,
    importe_neto_entregado_clp,
    ticket_promedio_clp,
    variacion_pedidos_mom_pct,
    variacion_importe_mom_pct,
    promedio_movil_importe_3m
FROM comparacion_mensual
ORDER BY importe_neto_entregado_clp DESC
LIMIT 5;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

anio_mes,pedidos_entregados,importe_neto_entregado_clp,ticket_promedio_clp,variacion_pedidos_mom_pct,variacion_importe_mom_pct,promedio_movil_importe_3m
2025-12,695,202353312,291155.84,44.49,39.34,164689686.67
2026-09,565,177839268,314759.77,15.54,17.08,158248456.67
2024-12,571,170933214,299357.64,27.46,30.96,142470050.0
2026-01,535,167012562,312173.01,-23.02,-17.46,171528162.0
2025-09,531,162198376,305458.34,16.7,19.21,139110202.0


In [120]:
%%sql

WITH resumen_mensual AS (
SELECT
    anio_mes,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.costo_linea_clp) AS costo_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp
FROM fact_detalle_pedidos AS t
LEFT JOIN dim_fecha AS f
    ON t.fecha_pedido = f.fecha
WHERE t.estado = 'entregado'
GROUP BY anio_mes
), 
comparacion_mensual AS (
SELECT
    anio_mes,
    pedidos_entregados,
    importe_neto_entregado_clp,
    ROUND(
        1.0 * importe_neto_entregado_clp / NULLIF(pedidos_entregados, 0)
        ,2) AS ticket_promedio_clp,
    ROUND(
        100.0 * (pedidos_entregados - LAG(pedidos_entregados) OVER (ORDER BY anio_mes) ) /  
        NULLIF(LAG(pedidos_entregados) OVER (ORDER BY anio_mes) , 0)
        , 2) AS variacion_pedidos_mom_pct,
    ROUND(
        100.0 * (importe_neto_entregado_clp - LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes) ) /  
        NULLIF(LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes), 0)
        , 2) AS variacion_importe_mom_pct,
   ROUND(
        AVG(importe_neto_entregado_clp) OVER(
        ORDER BY anio_mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
        , 2) AS promedio_movil_importe_3m
FROM resumen_mensual
)
SELECT
    anio_mes,
    pedidos_entregados,
    importe_neto_entregado_clp,
    ticket_promedio_clp,
    variacion_pedidos_mom_pct,
    variacion_importe_mom_pct,
    promedio_movil_importe_3m
FROM comparacion_mensual
ORDER BY importe_neto_entregado_clp ASC
LIMIT 5;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

anio_mes,pedidos_entregados,importe_neto_entregado_clp,ticket_promedio_clp,variacion_pedidos_mom_pct,variacion_importe_mom_pct,promedio_movil_importe_3m
2024-02,307,92063024,299879.56,-30.54,-26.79,108903476.0
2024-06,334,96920786,290181.99,-11.17,-14.63,105196026.67
2025-02,350,101192626,289121.79,-25.53,-27.47,137215502.0
2024-04,365,105131436,288031.33,-9.43,-10.2,104754864.0
2024-07,366,111328472,304176.15,9.58,14.87,107261705.33


In [121]:
%%sql 

WITH resumen_mensual AS (
SELECT
    anio_mes,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.costo_linea_clp) AS costo_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp
FROM fact_detalle_pedidos AS t
LEFT JOIN dim_fecha AS f
    ON t.fecha_pedido = f.fecha
WHERE t.estado = 'entregado'
GROUP BY anio_mes
), 
comparacion_mensual AS (
SELECT
    anio_mes,
    pedidos_entregados,
    importe_neto_entregado_clp,
    ROUND(
        1.0 * importe_neto_entregado_clp / NULLIF(pedidos_entregados, 0)
        ,2) AS ticket_promedio_clp,
    ROUND(
        100.0 * (pedidos_entregados - LAG(pedidos_entregados) OVER (ORDER BY anio_mes) ) /  
        NULLIF(LAG(pedidos_entregados) OVER (ORDER BY anio_mes) , 0)
        , 2) AS variacion_pedidos_mom_pct,
    ROUND(
        100.0 * (importe_neto_entregado_clp - LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes) ) /  
        NULLIF(LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes), 0)
        , 2) AS variacion_importe_mom_pct,
   ROUND(
        AVG(importe_neto_entregado_clp) OVER(
        ORDER BY anio_mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
        , 2) AS promedio_movil_importe_3m
FROM resumen_mensual
)
SELECT *
FROM comparacion_mensual
WHERE variacion_importe_mom_pct IS NOT NULL
ORDER BY variacion_importe_mom_pct DESC
LIMIT 5;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

anio_mes,pedidos_entregados,importe_neto_entregado_clp,ticket_promedio_clp,variacion_pedidos_mom_pct,variacion_importe_mom_pct,promedio_movil_importe_3m
2025-03,460,142726588,310275.19,31.43,41.04,127813293.33
2025-12,695,202353312,291155.84,44.49,39.34,164689686.67
2024-12,571,170933214,299357.64,27.46,30.96,142470050.0
2026-03,510,151477356,297014.42,34.56,28.52,145452443.33
2024-03,403,117070132,290496.61,31.27,27.16,111625694.67


In [122]:
%%sql 

WITH resumen_mensual AS (
SELECT
    anio_mes,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.costo_linea_clp) AS costo_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp
FROM fact_detalle_pedidos AS t
LEFT JOIN dim_fecha AS f
    ON t.fecha_pedido = f.fecha
WHERE t.estado = 'entregado'
GROUP BY anio_mes
), 
comparacion_mensual AS (
SELECT
    anio_mes,
    pedidos_entregados,
    importe_neto_entregado_clp,
    ROUND(
        1.0 * importe_neto_entregado_clp / NULLIF(pedidos_entregados, 0)
        ,2) AS ticket_promedio_clp,
    ROUND(
        100.0 * (pedidos_entregados - LAG(pedidos_entregados) OVER (ORDER BY anio_mes) ) /  
        NULLIF(LAG(pedidos_entregados) OVER (ORDER BY anio_mes) , 0)
        , 2) AS variacion_pedidos_mom_pct,
    ROUND(
        100.0 * (importe_neto_entregado_clp - LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes) ) /  
        NULLIF(LAG(importe_neto_entregado_clp) OVER(ORDER BY anio_mes), 0)
        , 2) AS variacion_importe_mom_pct,
   ROUND(
        AVG(importe_neto_entregado_clp) OVER(
        ORDER BY anio_mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
        , 2) AS promedio_movil_importe_3m
FROM resumen_mensual
)
SELECT *
FROM comparacion_mensual
WHERE variacion_importe_mom_pct IS NOT NULL
ORDER BY variacion_importe_mom_pct ASC
LIMIT 5;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

anio_mes,pedidos_entregados,importe_neto_entregado_clp,ticket_promedio_clp,variacion_pedidos_mom_pct,variacion_importe_mom_pct,promedio_movil_importe_3m
2026-02,379,117867412,310995.81,-29.16,-29.43,162411095.33
2025-02,350,101192626,289121.79,-25.53,-27.47,137215502.0
2024-02,307,92063024,299879.56,-30.54,-26.79,108903476.0
2025-04,400,115854286,289635.72,-13.04,-18.83,119924500.0
2025-01,470,139520666,296852.48,-17.69,-18.38,146993770.67



### 3.1.2 Evolución mensual y períodos destacados

#### Metodología

Se calcularon indicadores mensuales de pedidos entregados, importe neto, ticket promedio y promedio móvil de tres meses.

Mediante `LAG()` se obtuvieron las variaciones intermensuales de pedidos e importes netos. Posteriormente se identificaron los cinco períodos con mayores y menores niveles de importe neto, así como las cinco variaciones positivas y negativas más pronunciadas.

#### Hallazgos

**Niveles de actividad**

- Diciembre de 2025 registró el mayor importe neto del período, con $202.353.312 CLP y 695 pedidos entregados.
- Febrero de 2024 presentó el menor importe neto, con $92.063.024 CLP y 307 pedidos entregados.
- Un ticket promedio elevado no implica necesariamente un mayor importe neto mensual, ya que también influye la cantidad de pedidos.

**Variaciones intermensuales**

- Marzo de 2025 registró el mayor crecimiento intermensual del importe neto (+41,04 %).
- Febrero de 2026 presentó la mayor contracción intermensual (-29,43 %).
- Febrero registró disminuciones destacadas en 2024, 2025 y 2026.
- Marzo presentó recuperaciones importantes en los tres años analizados.
- Diciembre mostró crecimientos destacados en 2024 y 2025.

#### Interpretación

Los resultados sugieren patrones temporales recurrentes que justifican un análisis adicional de estacionalidad.

Las variaciones del importe neto deben interpretarse conjuntamente con la cantidad de pedidos y el ticket promedio.

Los cambios observados no permiten establecer por sí solos sus causas. La distinta cantidad de días calendario y hábiles entre meses, así como la composición comercial, podrían influir en las comparaciones.

#### Limitaciones

Los períodos se agrupan según la fecha de creación de pedidos actualmente entregados, no según su fecha efectiva de entrega.

Los importes no representan necesariamente facturación emitida ni cobros realizados.

Los datos son ficticios y los patrones identificados requieren verificaciones adicionales antes de formular recomendaciones comerciales.

#### Conclusión

El análisis permitió identificar períodos extremos y fluctuaciones relevantes en la actividad comercial.

Los resultados proporcionan una base para investigar patrones temporales y profundizar en los factores asociados al desempeño de clientes, segmentos y productos.



## 3.2 Análisis de clientes y segmentos

### 3.2.1 Actividad comercial de la cartera de clientes

#### Objetivo

Evaluar la composición y actividad comercial de la cartera de clientes de Distribuidora Cordillera, distinguiendo entre clientes registrados, clientes con pedidos entregados y clientes sin actividad registrada.

#### Metodología

Se utilizarán las vistas `dim_cliente` y `fact_detalle_pedidos`, considerando el período completo disponible en la base de datos.

Se definirá como cliente activo a aquel que registre al menos un pedido en estado `entregado`.

El análisis se desarrollará por segmento comercial y permitirá identificar diferencias en la proporción de clientes activos.

#### Consideraciones

La ausencia de pedidos entregados no implica necesariamente que un cliente haya abandonado la empresa.

Los resultados representan actividad comercial registrada durante el período analizado y no permiten determinar por sí solos retención, abandono o pérdida definitiva de clientes.


In [123]:
%%sql

SELECT
    c.segmento,
    COUNT(DISTINCT c.cliente_id) AS clientes_registrados,
    COUNT(DISTINCT t.cliente_id) AS clientes_activos,
    COUNT(DISTINCT c.cliente_id) - COUNT(DISTINCT t.cliente_id) AS clientes_sin_actividad,
    ROUND(100.0 * COUNT(DISTINCT t.cliente_id) / COUNT(DISTINCT c.cliente_id) , 2) AS tasa_actividad_pct
FROM dim_cliente AS c 
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = "entregado"
GROUP BY c.segmento
ORDER BY tasa_actividad_pct DESC;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

segmento,clientes_registrados,clientes_activos,clientes_sin_actividad,tasa_actividad_pct
Minimarket,159,150,9,94.34
Horeca,110,103,7,93.64
Almacen,331,309,22,93.35



#### Resultados y hallazgos

| Segmento | Clientes registrados | Clientes activos | Sin pedidos entregados | Tasa de actividad |
|---|---:|---:|---:|---:|
| Minimarket | 159 | 150 | 9 | 94,34 % |
| Horeca | 110 | 103 | 7 | 93,64 % |
| Almacen | 331 | 309 | 22 | 93,35 % |
| **Total** | **600** | **562** | **38** | **93,67 %** |

Se identificaron 562 clientes con al menos un pedido entregado durante el período analizado, equivalentes al 93,67 % de la cartera registrada.

Minimarket presenta la mayor tasa de actividad histórica, mientras que Almacen concentra la mayor cantidad absoluta de clientes sin pedidos entregados.

Las diferencias porcentuales entre segmentos son reducidas.

**Limitación:** la clasificación de actividad considera el período histórico completo. Por tanto, no permite determinar si los clientes mantienen actividad reciente, si presentan recurrencia comercial o si han dejado de comprar.

**Conclusión:** la cartera presenta una elevada proporción de clientes con actividad histórica registrada. Se requiere profundizar en frecuencia, temporalidad e importes para evaluar el comportamiento de compra.


In [124]:
%%sql

SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp,
    ROUND(1.0 * SUM(t.importe_neto_linea_clp) / NULLIF(COUNT(DISTINCT t.pedido_id),0), 2) AS ticket_promedio_clp
FROM dim_cliente AS c
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
ORDER BY importe_neto_entregado_clp DESC
;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

cliente_id,razon_social,segmento,pedidos_entregados,importe_neto_entregado_clp,ticket_promedio_clp
52,Almacen Ficticio Cordillera 0052,Almacen,156,48125814,308498.81
67,Almacen Ficticio Cordillera 0067,Almacen,153,45871116,299811.22
57,Almacen Ficticio Cordillera 0057,Almacen,130,45676530,351357.92
48,Minimarket Ficticio Cordillera 0048,Minimarket,144,45091854,313137.88
54,Minimarket Ficticio Cordillera 0054,Minimarket,145,43859664,302480.44
63,Almacen Ficticio Cordillera 0063,Almacen,133,43667496,328327.04
74,Almacen Ficticio Cordillera 0074,Almacen,128,43276812,338100.09
73,Almacen Ficticio Cordillera 0073,Almacen,126,42543000,337642.86
72,Almacen Ficticio Cordillera 0072,Almacen,134,42428298,316629.09
64,Almacen Ficticio Cordillera 0064,Almacen,131,42312942,322999.56


In [125]:
%%sql

WITH resumen_clientes AS (
SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp,
    ROUND(1.0 * SUM(t.importe_neto_linea_clp) / NULLIF(COUNT(DISTINCT t.pedido_id),0), 2) AS ticket_promedio_clp
FROM dim_cliente AS c
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
)
SELECT
    DENSE_RANK() OVER( ORDER BY importe_neto_entregado_clp DESC
    ) AS ranking_cliente,
    *
FROM resumen_clientes
ORDER BY importe_neto_entregado_clp DESC
LIMIT 10;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

ranking_cliente,cliente_id,razon_social,segmento,pedidos_entregados,importe_neto_entregado_clp,ticket_promedio_clp
1,52,Almacen Ficticio Cordillera 0052,Almacen,156,48125814,308498.81
2,67,Almacen Ficticio Cordillera 0067,Almacen,153,45871116,299811.22
3,57,Almacen Ficticio Cordillera 0057,Almacen,130,45676530,351357.92
4,48,Minimarket Ficticio Cordillera 0048,Minimarket,144,45091854,313137.88
5,54,Minimarket Ficticio Cordillera 0054,Minimarket,145,43859664,302480.44
6,63,Almacen Ficticio Cordillera 0063,Almacen,133,43667496,328327.04
7,74,Almacen Ficticio Cordillera 0074,Almacen,128,43276812,338100.09
8,73,Almacen Ficticio Cordillera 0073,Almacen,126,42543000,337642.86
9,72,Almacen Ficticio Cordillera 0072,Almacen,134,42428298,316629.09
10,64,Almacen Ficticio Cordillera 0064,Almacen,131,42312942,322999.56



### 3.2.2 Importe neto y frecuencia de pedidos por cliente

#### Metodología

Se consolidaron los pedidos entregados e importes netos a nivel de cliente, utilizando `dim_cliente` y `fact_detalle_pedidos`.

Se conservaron los 600 clientes registrados, incluidos aquellos sin pedidos entregados.

Para evitar duplicaciones, los pedidos se contabilizaron mediante `COUNT(DISTINCT pedido_id)` y los importes se calcularon desde las líneas de detalle.

El ticket promedio se definió como el importe neto entregado dividido por la cantidad de pedidos entregados de cada cliente.

### 3.2.3 Ranking comercial de clientes

Se construyó un ranking de clientes según el importe neto asociado a pedidos entregados, utilizando `DENSE_RANK()`.

#### Hallazgos preliminares

- Los primeros diez puestos están integrados por ocho clientes Almacen y dos clientes Minimarket.
- Los clientes de mayor importe neto presentan cantidades elevadas de pedidos entregados durante el período analizado.
- El importe neto individual depende tanto de la cantidad de pedidos como de su ticket promedio.

#### Consideraciones

La posición comercial por importe neto no equivale necesariamente a una mayor rentabilidad.

La concentración del importe neto entre clientes se evaluará posteriormente mediante indicadores de participación y análisis Pareto.

Los resultados corresponden al período histórico disponible y no permiten determinar por sí solos la actividad reciente ni la fidelización de los clientes.



### 3.2.4 Distribución de la frecuencia histórica de compra

#### Objetivo

Caracterizar la cartera comercial según la cantidad de pedidos entregados registrados por cliente durante el período analizado.

#### Metodología

Se utilizará el resumen comercial a nivel de cliente construido anteriormente.

Los clientes se clasificarán en cinco rangos según la cantidad de pedidos entregados, conservando también a aquellos sin actividad.

Para cada rango se calculará la cantidad de clientes, su participación dentro de la cartera, el total de pedidos entregados y el importe neto asociado.

#### Consideraciones

Los rangos definidos tienen carácter exploratorio.

La cantidad histórica de pedidos no representa necesariamente una frecuencia mensual o anual comparable entre clientes, especialmente cuando sus períodos de antigüedad comercial son diferentes.

Los resultados permitirán describir la distribución de actividad, pero no determinar por sí solos fidelización o retención.


In [126]:
%%sql

WITH resumen_clientes AS (
SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    COUNT(DISTINCT t.pedido_id) AS pedidos_entregados,
    COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp,
    ROUND(1.0 * SUM(t.importe_neto_linea_clp) / NULLIF(COUNT(DISTINCT t.pedido_id),0), 2) AS ticket_promedio_clp
FROM dim_cliente AS c
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
),
clasificacion_clientes AS (
SELECT
    *,
    CASE
        WHEN pedidos_entregados = 0 THEN 'Sin actividad'
        WHEN pedidos_entregados BETWEEN 1 AND 10 THEN '1-10'
        WHEN pedidos_entregados BETWEEN 11 AND 30 THEN '11-30'
        WHEN pedidos_entregados BETWEEN 31 AND 60 THEN '31-60'
        WHEN pedidos_entregados > 60 THEN '61+'
        ELSE NULL 
    END AS rango_pedidos,
    CASE
        WHEN pedidos_entregados = 0 THEN 1
        WHEN pedidos_entregados BETWEEN 1 AND 10 THEN 2
        WHEN pedidos_entregados BETWEEN 11 AND 30 THEN 3
        WHEN pedidos_entregados BETWEEN 31 AND 60 THEN 4
        WHEN pedidos_entregados > 60 THEN 5
    END AS orden_rango
    FROM resumen_clientes
)
SELECT
    rango_pedidos,
    COUNT(*) AS total_clientes,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct_cartera,
    SUM(pedidos_entregados) AS total_pedidos_entregados,
    SUM(importe_neto_entregado_clp) AS importe_neto_entregado_clp
FROM clasificacion_clientes
GROUP BY rango_pedidos
ORDER BY orden_rango ASC
;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

rango_pedidos,total_clientes,pct_cartera,total_pedidos_entregados,importe_neto_entregado_clp
Sin actividad,38,6.33,0,0
1-10,236,39.33,1371,412649222
11-30,130,21.67,1861,580756112
31-60,153,25.5,6872,2047487612
61+,43,7.17,4630,1393309088



#### Resultados y hallazgos

| Frecuencia histórica | Clientes | % cartera | Pedidos entregados | Importe neto (CLP) |
|---|---:|---:|---:|---:|
| Sin actividad | 38 | 6,33 % | 0 | 0 |
| 1–10 pedidos | 236 | 39,33 % | 1.371 | 412.649.222 |
| 11–30 pedidos | 130 | 21,67 % | 1.861 | 580.756.112 |
| 31–60 pedidos | 153 | 25,50 % | 6.872 | 2.047.487.612 |
| 61+ pedidos | 43 | 7,17 % | 4.630 | 1.393.309.088 |
| **Total** | **600** | **100 %** | **14.734** | **4.434.202.034** |

Los 196 clientes que registraron más de 30 pedidos representan el 32,67 % de la cartera y concentran el 77,60 % del importe neto asociado a pedidos entregados.

En contraste, los 236 clientes con entre 1 y 10 pedidos representan el 39,33 % de la cartera, pero aportan únicamente el 9,31 % del importe neto.

#### Interpretación comercial

Se observa una asociación relevante entre la cantidad histórica de pedidos y el aporte monetario de los clientes.

La distribución sugiere que los compradores de mayor frecuencia desempeñan un papel importante en el resultado comercial agregado.

Sin embargo, estos resultados no permiten determinar por sí solos fidelización, recurrencia reciente o rentabilidad individual.

#### Conclusión de la sección 3.2

La cartera comercial presenta una elevada proporción de clientes con actividad histórica, aunque el importe neto se distribuye de manera desigual entre los distintos niveles de frecuencia.

El análisis permite identificar grupos de clientes relevantes para posteriores estudios de concentración, recurrencia y rentabilidad bruta estimada.

Los resultados servirán como referencia para el análisis de dependencia comercial que se desarrollará en el Notebook 04.



## 3.3 Desempeño de productos y categorías

### 3.3.1 Indicadores comerciales por categoría

#### Objetivo

Evaluar el desempeño de las categorías de productos según volumen entregado, importe neto y margen bruto estimado.

#### Metodología

Se relacionarán las vistas `dim_producto` y `fact_detalle_pedidos` mediante `producto_id`.

Se considerarán exclusivamente pedidos en estado `entregado` para los indicadores comerciales, conservando los productos del catálogo.

Los resultados se agruparán por categoría de producto y permitirán comparar la contribución monetaria y el margen bruto estimado de cada grupo.

#### Consideraciones

La cantidad de unidades corresponde al campo `cantidad` registrado en las líneas de pedido.

Los márgenes se calcularán utilizando los importes y costos registrados en las transacciones, no los precios y costos actuales del catálogo.

La clasificación por categoría corresponde a los atributos disponibles en `dim_producto`.


In [127]:
%%sql

SELECT
    p.nombre_categoria,
    COUNT(DISTINCT p.producto_id) AS productos_catalogo,
    COUNT(DISTINCT t.producto_id) AS productos_con_entregas,
    SUM(t.cantidad) AS unidades_entregadas,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen__bruto_estimado_clp,
    ROUND(100.0 * SUM(t.margen_bruto_estimado_linea_clp) / NULLIF(SUM(t.importe_neto_linea_clp), 0), 2) AS pct_margen_bruto_estimado
FROM dim_producto AS p
LEFT JOIN fact_detalle_pedidos AS t 
    ON p.producto_id = t.producto_id
    AND t.estado = 'entregado'
GROUP BY nombre_categoria
ORDER BY importe_neto_entregado_clp DESC

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

nombre_categoria,productos_catalogo,productos_con_entregas,unidades_entregadas,importe_neto_entregado_clp,margen__bruto_estimado_clp,pct_margen_bruto_estimado
Abarrotes,30,30,485754,1671546730,492181044,29.44
Bebidas,30,30,215200,738422998,213072712,28.86
Limpieza,30,30,160518,561189938,149568244,26.65
Snacks,30,30,163302,553764834,162029984,29.26
Desechables,30,30,163190,512373440,131029200,25.57
Cafeteria,30,20,128730,396904094,114221564,28.78



#### Resultados y hallazgos

El análisis contempla seis categorías comerciales y un catálogo de 180 productos.

El importe neto total asociado a pedidos entregados alcanza $4.434.202.034 CLP, coincidiendo con los resultados previamente conciliados.

**Principales hallazgos**

- Abarrotes concentra el 37,70 % del importe neto entregado y presenta el mayor margen bruto porcentual estimado (29,44 %).
- Desechables registra el menor margen bruto porcentual estimado (25,57 %).
- Cafetería presenta 20 productos con entregas de un total de 30 registrados.
- Las otras cinco categorías presentan entregas asociadas a todos sus productos.

#### Interpretación comercial

Se observan diferencias relevantes entre las categorías en términos de participación monetaria, cobertura comercial del catálogo y margen bruto estimado.

La importancia económica de una categoría no debe evaluarse únicamente mediante su volumen de unidades, sino también considerando los importes netos y márgenes asociados.

La ausencia de entregas en determinados productos no constituye automáticamente evidencia de obsolescencia o bajo desempeño estructural.

#### Conclusión

El análisis identifica diferencias en la contribución comercial de las categorías y permite seleccionar áreas para investigaciones posteriores.

Se recomienda profundizar en el desempeño individual de los productos y reservar la evaluación detallada de rentabilidad para el Notebook 04.


In [128]:
%%sql

SELECT
    p.nombre_producto,
    p.nombre_categoria,
    SUM(t.cantidad) AS unidades_entregadas,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    ROUND(
        1.0 * SUM(t.importe_neto_linea_clp) / NULLIF(SUM(t.cantidad),0), 2
    ) AS precio_neto_promedio_clp,
    ROUND(
        100.0 * SUM(t.margen_bruto_estimado_linea_clp) / NULLIF(SUM(t.importe_neto_linea_clp),0) , 2
    ) AS pct_margen_bruto_estimado
FROM dim_producto AS p
LEFT JOIN fact_detalle_pedidos AS t 
    ON p.producto_id = t.producto_id
    AND t.estado = 'entregado' 
GROUP BY p.producto_id, p.nombre_producto, p.nombre_categoria
ORDER BY importe_neto_entregado_clp DESC
LIMIT 10

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

nombre_producto,nombre_categoria,unidades_entregadas,importe_neto_entregado_clp,precio_neto_promedio_clp,pct_margen_bruto_estimado
Azucar 1 kg Marca Ficticia 1,Abarrotes,20670,128699986,6226.41,34.17
Aceite 1 L Marca Ficticia 1,Abarrotes,21988,124568346,5665.29,35.07
Fideos 400 g Marca Ficticia 4,Abarrotes,21306,120398184,5650.91,33.93
Aceite 1 L Marca Ficticia 3,Abarrotes,22120,119916960,5421.2,31.12
Harina 1 kg Marca Ficticia 2,Abarrotes,22062,109911434,4981.93,21.87
Arroz 1 kg Marca Ficticia 1,Abarrotes,21692,94364198,4350.18,21.14
Harina 1 kg Marca Ficticia 1,Abarrotes,21582,94029090,4356.83,27.09
Arroz 1 kg Marca Ficticia 2,Abarrotes,20932,85429362,4081.28,34.83
Fideos 400 g Marca Ficticia 3,Abarrotes,20736,75664086,3648.92,32.7
Harina 1 kg Marca Ficticia 3,Abarrotes,21842,69476608,3180.87,34.0



#### Resultados y hallazgos

El análisis identificó los diez productos con mayor importe neto asociado a pedidos entregados, considerando unidades, precio neto promedio ponderado y margen bruto estimado.

**Principales hallazgos**

- Los diez productos de mayor importe neto pertenecen a la categoría Abarrotes.
- Azúcar 1 kg Marca Ficticia 1 lidera el ranking con aproximadamente $128,7 millones CLP y 20.670 unidades entregadas.
- Los volúmenes de los diez productos son relativamente similares, pero existen diferencias relevantes en sus precios netos promedio.
- Los márgenes brutos estimados del Top 10 oscilan entre 21,14 % y 35,07 %.

#### Interpretación comercial

El desempeño monetario individual no depende exclusivamente del volumen de unidades entregadas. El precio neto promedio ponderado también influye en la generación de importe neto.

Los productos con mayores importes no necesariamente presentan los mayores márgenes porcentuales.

La presencia exclusiva de Abarrotes en el Top 10 refuerza su relevancia comercial, aunque otras categorías también contribuyen al resultado agregado.

#### Conclusión de la sección 3.3

El análisis de categorías y productos permitió identificar diferencias en participación comercial, cobertura del catálogo, volumen, precios netos y márgenes brutos estimados.

Estos resultados constituyen una base para evaluar posteriormente la concentración de ingresos y la rentabilidad bruta estimada a nivel de producto y categoría.

La evaluación de oportunidades comerciales específicas se reservará para el Notebook 04.



## 3.4 Síntesis ejecutiva del desempeño comercial

### 3.4.1 Resumen de indicadores

El análisis del desempeño comercial de Distribuidora Cordillera se desarrolló a partir del modelo dimensional construido y validado previamente.

Se evaluaron la evolución temporal, la actividad de clientes, la frecuencia histórica de compra y el desempeño de productos y categorías.

Los indicadores monetarios consideran exclusivamente pedidos registrados en estado `entregado`.

| Indicador | Resultado |
|---|---:|
| Pedidos entregados | 14.734 |
| Importe neto entregado | $4.434.202.034 CLP |
| Costo asociado | $3.172.099.286 CLP |
| Margen bruto estimado | $1.262.102.748 CLP |
| Margen bruto estimado (%) | 28,46 % |
| Clientes registrados | 600 |
| Clientes con actividad histórica | 562 |
| Productos del catálogo | 180 |

### 3.4.2 Principales hallazgos

**1. Evolución temporal**

Se identificaron fluctuaciones relevantes en la actividad comercial entre enero de 2024 y septiembre de 2026.

Diciembre de 2025 presentó el mayor importe neto mensual, con aproximadamente $202,35 millones CLP.

Febrero registró contracciones intermensuales destacadas en los tres años analizados, mientras que marzo presentó recuperaciones importantes en esos mismos períodos.

Estos comportamientos sugieren posibles patrones temporales recurrentes, cuya explicación requiere investigaciones adicionales.

**2. Actividad de clientes**

De los 600 clientes registrados, 562 presentaron al menos un pedido entregado durante el período histórico, equivalentes al 93,67 % de la cartera.

Las tasas de actividad histórica fueron similares entre los segmentos Minimarket, Horeca y Almacen.

La actividad histórica no permite determinar por sí sola si los clientes mantienen relaciones comerciales activas en la actualidad.

**3. Distribución de la frecuencia de compra**

Los 196 clientes con más de 30 pedidos representaron el 32,67 % de la cartera y concentraron aproximadamente el 77,60 % del importe neto entregado.

En contraste, los 236 clientes con entre 1 y 10 pedidos representaron el 39,33 % de la cartera y aportaron aproximadamente el 9,31 % del importe neto.

Se observa una asociación importante entre frecuencia histórica de pedidos y contribución monetaria.

**4. Desempeño por categoría**

Abarrotes concentró aproximadamente el 37,70 % del importe neto entregado y presentó un margen bruto estimado del 29,44 %.

Desechables registró el menor margen porcentual estimado entre las categorías, con un 25,57 %.

Cafetería presentó 20 productos con entregas de un total de 30 productos registrados.

**5. Desempeño individual de productos**

Los diez productos con mayor importe neto pertenecieron a Abarrotes.

Se observaron diferencias relevantes entre sus precios netos promedio ponderados y sus márgenes porcentuales estimados.

Esto demuestra que el volumen de unidades no explica por sí solo la contribución monetaria de cada producto.

---

## 3.5 Conclusiones e implicaciones comerciales

### 3.5.1 Conclusiones principales

**Primera conclusión: el desempeño comercial presenta variaciones temporales relevantes.**

La evolución mensual muestra diferencias en el volumen de pedidos y el importe neto entregado.

Los cambios del importe neto deben evaluarse conjuntamente con la cantidad de pedidos y el ticket promedio para comprender sus componentes.

**Segunda conclusión: la actividad comercial presenta diferencias entre grupos de clientes.**

Aunque la mayoría de los clientes registra actividad histórica, los grupos con mayor frecuencia de pedidos concentran una proporción elevada del importe neto.

Este comportamiento justifica profundizar en la concentración comercial a nivel individual y en la recurrencia temporal.

**Tercera conclusión: los productos y categorías presentan contribuciones heterogéneas.**

Abarrotes destaca por su importancia monetaria y por la presencia de sus productos en los primeros puestos del ranking comercial.

Sin embargo, las diferencias de margen observadas muestran que el volumen o el importe neto no deben utilizarse como únicos criterios para evaluar el desempeño de productos y categorías.

### 3.5.2 Oportunidades de investigación

A partir de los resultados, se identifican las siguientes líneas de análisis:

1. **Comportamiento temporal:** evaluar la recurrencia de las variaciones de febrero y marzo, considerando duración de los meses y días de actividad comercial.
2. **Concentración comercial:** determinar qué proporción del importe neto depende de los principales clientes y productos.
3. **Recurrencia de clientes:** diferenciar actividad histórica de comportamiento reciente mediante indicadores temporales.
4. **Rentabilidad:** comparar la contribución monetaria y los márgenes brutos estimados entre clientes, productos y categorías.
5. **Cobertura del catálogo:** investigar productos sin entregas registradas y su relevancia comercial.

Estas oportunidades constituyen hipótesis y líneas de investigación. No representan todavía recomendaciones operativas definitivas.

---

## 3.6 Limitaciones y continuidad del análisis

### Limitaciones

**Naturaleza de los datos**

El proyecto utiliza información ficticia de una distribuidora comercial. Sus resultados tienen fines analíticos y de demostración metodológica.

**Definición monetaria**

Los importes netos corresponden a pedidos registrados en estado `entregado`. No deben interpretarse automáticamente como ventas facturadas ni ingresos efectivamente cobrados.

**Rentabilidad**

Los márgenes brutos estimados se calculan a partir de los importes y costos registrados en las líneas de pedido.

No incorporan necesariamente gastos operacionales, financieros, tributarios ni otros componentes requeridos para determinar la utilidad neta.

**Temporalidad**

La evolución mensual utiliza la fecha de creación de pedidos actualmente entregados, no su fecha efectiva de entrega.

**Actividad de clientes**

Los indicadores de frecuencia consideran el período histórico completo. Las diferencias en antigüedad comercial pueden afectar la comparabilidad entre clientes.

**Causalidad**

Las relaciones y patrones identificados son descriptivos. No permiten establecer causalidad sin análisis y evidencia adicionales.

### Continuidad del proyecto

Los hallazgos obtenidos constituyen la base para desarrollar el siguiente componente:

**Notebook 04 — Concentración comercial y rentabilidad bruta estimada**

Este análisis profundizará en:

- Concentración del importe neto por clientes y productos.
- Clasificación ABC y análisis Pareto.
- Dependencia comercial de clientes relevantes.
- Comparación de márgenes brutos estimados.
- Identificación de segmentos y productos que ameritan revisión comercial.
- Formulación de recomendaciones sustentadas en indicadores.

Posteriormente, los resultados se integrarán en un modelo semántico de Power BI para construir indicadores DAX y visualizaciones ejecutivas.

### Conclusión general

El Notebook 03 permitió transformar el modelo dimensional previamente implementado en información analítica sobre el comportamiento comercial de Distribuidora Cordillera.

Mediante SQL avanzado, agregaciones, CTE y funciones de ventana, se identificaron patrones temporales, diferencias en la actividad de clientes y variaciones en el desempeño de productos y categorías.

Los resultados proporcionan una base documentada para profundizar en concentración comercial, rentabilidad bruta estimada y desarrollo de soluciones de Business Intelligence.

**Estado del análisis:** finalizado.

